# Olist Brazilian E-Commerce — Business Intelligence Analysis

## Project Overview
This notebook starts v0.1.0: **Data Cleaning & Data Integration**. The Olist dataset is relational, so we first inspect and validate each table before deciding how to join them.

**Execution environment:** GitHub Codespaces / VS Code. This notebook does not use Google Colab or Google Drive.


## 1. Setup Environment
Import the libraries needed for data manipulation and initial visualization.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Pandas version: {pd.__version__}')


## 2. Project Paths
Resolve the repository root so the notebook works whether the Codespace opens the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'notebooks').exists():
            return path
    raise FileNotFoundError(
        'Project root not found. Open this notebook inside the olist-bi-project repository.'
    )

ROOT = find_project_root()
RAW_DATA = ROOT / 'data' / 'raw'
PROCESSED_DATA = ROOT / 'data' / 'processed'

print(f'Project root : {ROOT}')
print(f'Raw data    : {RAW_DATA}')
print(f'Processed   : {PROCESSED_DATA}')


## 3. Load Datasets
The Olist dataset contains 9 CSV tables. The raw files must already be placed in `data/raw/` before running this section.


In [ ]:
file_names = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {}
missing_files = []

for name, filename in file_names.items():
    filepath = RAW_DATA / filename
    try:
        dfs[name] = pd.read_csv(filepath)
        print(f'Loaded {name:20s} | Shape: {dfs[name].shape}')
    except FileNotFoundError:
        missing_files.append(filename)
        print(f'File not found: {filename}')

print(f'\nTotal datasets loaded: {len(dfs)} / {len(file_names)}')
if missing_files:
    print(f'Missing files: {missing_files}')


## 4. Quick Overview per Table
Before cleaning anything, inspect row counts, columns, data types, sample records, and missing values. The findings here will determine the cleaning strategy for each table.


In [ ]:
for name, df in dfs.items():
    print('\n' + '=' * 70)
    print(f'DATASET: {name.upper()}')
    print('=' * 70)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(f'Columns: {list(df.columns)}')
    print('\nData types:')
    print(df.dtypes.value_counts().to_string())
    print('\nFirst 2 rows:')
    print(df.head(2))
    print('\nMissing values (%):')
    missing = (df.isna().mean() * 100).sort_values(ascending=False)
    print(missing[missing > 0].to_string() if (missing > 0).any() else 'No missing values')


### Interpretation

The initial profile shows that the 9 tables have different structures and data-quality characteristics. Several timestamp fields are currently loaded as strings, while missing values are concentrated in `orders`, `order_reviews`, and `products`.

In `orders`, missing delivery and approval timestamps should be interpreted together with `order_status` rather than filled automatically. In `order_reviews`, missing comment text does not necessarily invalidate the review because review scores and metadata may still be available. In `products`, missing attributes should be assessed according to how each field will be used in later analysis.

These findings indicate that cleaning should be evidence-based and table-specific rather than applying a single missing-value rule across the entire dataset.


## 5. Summary of All Tables


In [ ]:
table_info = pd.DataFrame({
    'Table': list(dfs.keys()),
    'Rows': [df.shape[0] for df in dfs.values()],
    'Columns': [df.shape[1] for df in dfs.values()]
})

table_info


### Interpretation

The 9 Olist tables have different row counts and therefore should not be assumed to share the same grain. `orders` contains 99,441 records, while `order_items` and `order_payments` contain more rows than `orders`. This indicates that these tables should not be treated as one-row-per-order tables without further validation.

The `geolocation` table is substantially larger than the transactional tables, with more than one million records. It should be treated as a separate location-level reference table until its grain and relationship to transactional data are validated.

These differences are important when designing the integration strategy because inappropriate joins could multiply rows and distort aggregated metrics.


## 6. Primary Key Uniqueness Check
Validate the identifiers that are expected to be unique before joining tables. A non-unique key can multiply rows during a merge.


In [ ]:
pk_checks = {
    'orders': 'order_id',
    'customers': 'customer_id',
    'sellers': 'seller_id',
    'products': 'product_id'
}

for table, pk in pk_checks.items():
    if table in dfs:
        df = dfs[table]
        n_rows = len(df)
        n_unique = df[pk].nunique(dropna=True)
        n_missing = df[pk].isna().sum()
        duplicates = n_rows - n_unique - n_missing
        status = 'VALID' if duplicates == 0 and n_missing == 0 else 'REVIEW'

        print(
            f'{table:15s} | {pk:20s} | '
            f'{n_rows:7,} rows | {n_unique:7,} unique | '
            f'{n_missing:5,} missing | {duplicates:5,} duplicates | '
            f'{status}'
        )


### Interpretation

The expected entity identifiers in `orders`, `customers`, `sellers`, and `products` are unique, with no missing or duplicate key values detected in the initial assessment.

This supports using these identifiers as entity-level keys when validating relationships with other tables. However, primary-key uniqueness does not prove that the entire table is free of duplicate rows; full-row duplicate validation remains a separate data-quality check.


## 7. Cross-Table Relationship Validation
Start with a basic relationship check: every `order_id` in `order_items` should have a corresponding record in `orders`. Additional relationships will be validated after we understand the actual table grains.


In [ ]:
if {'orders', 'order_items'}.issubset(dfs):
    orders_ids = set(dfs['orders']['order_id'])
    items_order_ids = set(dfs['order_items']['order_id'])

    matched_order_ids = items_order_ids & orders_ids
    orphan_order_ids = items_order_ids - orders_ids

    total_item_order_ids = len(items_order_ids)
    matched_count = len(matched_order_ids)
    orphan_count = len(orphan_order_ids)
    coverage = matched_count / total_item_order_ids * 100 if total_item_order_ids else 0

    print(f'Orders unique IDs           : {len(orders_ids):,}')
    print(f'Order item unique order IDs : {total_item_order_ids:,}')
    print(f'Matched order IDs           : {matched_count:,}')
    print(f'Orphan order IDs            : {orphan_count:,}')
    print(f'Relationship coverage       : {coverage:.2f}%')
else:
    print('Load orders and order_items before running this validation.')


### Interpretation

All distinct `order_id` values found in `order_items` are present in the `orders` table, resulting in 100% relationship coverage and zero orphan order IDs.

The number of unique order IDs in `order_items` is lower than the number of records in `orders`. This difference does not by itself indicate a data-quality problem because the validation is performed from the `order_items` side: every order referenced by an item must exist in the parent `orders` table.

This confirms that `order_id` is a valid relationship key for the validated `orders` → `order_items` relationship, while the two tables retain different grains.


## 8. Initial Findings & Cleaning Strategy

The initial assessment shows that the Olist dataset contains multiple tables with different grains, data types, and data-quality characteristics. The cleaning process will therefore be based on evidence observed in each table rather than applying the same transformation to all datasets.

### Initial Findings

- All 9 source tables were loaded successfully.
- The expected entity identifiers in `orders`, `customers`, `sellers`, and `products` are unique, with no missing or duplicate key values detected.
- The validated `orders` → `order_items` relationship has 100% coverage from `order_items` to `orders`, with no orphan `order_id` values detected.
- Several timestamp columns are currently stored as strings and require conversion to datetime.
- Missing values are concentrated in `orders`, `order_reviews`, and `products`, but they should not be removed or imputed automatically.
- `order_items` and `order_payments` contain more rows than `orders`, confirming that table grain must be considered during integration.
- The `geolocation` table contains more than one million records and requires separate grain validation before being used in analytical joins.

### Cleaning Strategy

Based on these findings, the next stage will focus on:

1. Converting relevant date and timestamp columns to appropriate datetime types.
2. Investigating missing values in `orders` according to order status and delivery lifecycle.
3. Preserving review records with missing comment text while retaining available review scores and metadata.
4. Investigating missing product attributes before deciding whether to retain, impute, or exclude specific fields.
5. Checking duplicate rows separately from primary-key uniqueness.
6. Validating additional cross-table relationships before performing analytical joins.
7. Preparing cleaned tables in `data/processed/` while preserving the original raw data.

### Integration Principle

No large multi-table merge will be performed until the grain, key, and relationship of the participating tables have been validated.
